In [2]:
!pip install deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.9 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=fc39a18b8e2667ad1ee812f103ef344aa7a84d8182c0d3af9dba5507683d5280
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


In [3]:
import threading
import cv2
from deepface import DeepFace

#Star Video Capture
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

#Load reference image
reference_img = cv2.imread('reference.jpg')

#Share variable
face_match = False
lock = threading.Lock()
counter = 0


def check_face(frame):
  global face_match
  try:
    result = DeepFace.verify(frame, reference_img.copy(), enforce_detection=False)
    if result['verified']:
      with lock:
        face_match = True
    else:
      with lock:
        face_match = False
  except ValueError as e:
    with lock:
      face_match = False
    print(f"Error verifying face: {e}")


while True:
  ret, frame = cap.read()
  if not ret:
    break

  #Check every 30 frames
  if counter % 30 == 0:
    threading.Thread(target=check_face, args=(frame.copy(),)).start()

  #Display match result
  with lock:
    if face_match:
      text = "MATCH !"
      color = (0, 255, 0)
    else:
      text = "NO MATCH !"
      color = (0, 0, 255)

  cv2.putText(frame, text, (20, 450), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
  cv2.imshow('Video', frame)

  key = cv2.waitKey(1)
  if key == ord("quit"):
    break


  counter += 1


cap.release()
cv2.destroyAllWindows()

25-05-19 13:45:30 - Directory /root/.deepface has been created
25-05-19 13:45:30 - Directory /root/.deepface/weights has been created
